### This is an experiment for new image-text retrieval web ###
NOTES:
- This lab uses old metadata and newly computed vectors to experiment proper grouping between metadatas and vectors

**SETUP**

In [3]:
import json
import csv
import torch
import os
from glob import glob
from pprint import pprint
from google.cloud import storage
from tqdm import tqdm
from PIL import Image

**LINK FETCHING**

In [10]:
http_prefix = 'https://storage.cloud.google.com/aic_2025/'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\Bui Thien Nghia\\gcs-user-key.json'
client = storage.Client()
bucket = client.get_bucket('aic_2025')
blobs = bucket.list_blobs(prefix='kfs/Keyframes_K')

all_img_keys = [blob.name for blob in blobs]
all_img_links = []
all_vid_links = {}

for img_key in all_img_keys:
    all_img_links.append(http_prefix + img_key)
    separations = img_key.split('/') 
    all_vid_links[separations[2]] = http_prefix + f'vids/{separations[1].replace('Keyframes', 'Videos')}/{separations[2]}.mp4'

print(f'Samples count: {len(all_img_links)}')

Samples count: 204978


**CSV LOADING**

In [11]:
csv_dict = {}
csv_file_paths = glob('D:\\AIC2025\\mapkfs\\b2\\*.csv')
for csv_path in csv_file_paths:
    video_id = csv_path.split('\\')[-1][:-4]
    csv_dict[video_id] = {}

    with open(csv_path, 'r', newline='') as csv_file:
        reader = csv.reader(csv_file)
        next(reader)

        for row in reader:
            csv_dict[video_id][f'{row[0]}'] = {
                'time_order': str(int(float(row[1]) * 1000)),
                'fps': str(row[2]),
                'frame_order': str(row[3])
            }

**JSON LOADING**

In [ ]:
json_dict = {}
json_file_paths = glob('D:\\AIC2025\\metadata\\b2\\*.json')
for json_path in json_file_paths:
    video_id = json_path.rsplit('\\', 1)[-1][:-5]
    with open(json_path, 'r', encoding='utf-8') as json_file:
        metadata = json.load(json_file)
    
    json_dict[video_id] = {
        'youtube_link': metadata['watch_url'],
        'publish_date': metadata['publish_date'] 
    }

**METADATA PACKAGING**

In [13]:
metadata_dict = {}
for img_link in all_img_links:
    video_id = img_link.split('/')[-2]
    vid_link = all_vid_links[video_id]
    frame_id = img_link.split('/')[-1][:-4]
    time_order = csv_dict[video_id][f'{int(frame_id)}']['time_order']
    frame_order = csv_dict[video_id][f'{int(frame_id)}']['frame_order']
    fps = csv_dict[video_id][f'{int(frame_id)}']['fps']
    answer_key = f'{video_id}-{time_order}'
    youtube_link = f'{json_dict[video_id]['youtube_link']}&t={time_order}ms'
    publish_date = json_dict[video_id]['publish_date']
    
    metadata_dict[f'{img_link}'] = {
        'img_link': img_link,
        'vid_link': vid_link,
        'vector': [],
        'video_id': video_id,
        'frame_id': frame_id,
        'time_order': time_order,
        'frame_order': frame_order,
        'fps': fps,
        'answer_key': answer_key,
        'youtube_link': youtube_link,
        'publish_date': publish_date
    }


**METADATA SAVING**

In [19]:
torch.save(metadata_dict, 'aic_2025-no-feat-b2.pt')

**DEBUGGING**

In [18]:
print(f'Total samples in dataset: {len(metadata_dict)}')
print('\nExample of a sample:')
pprint(list(metadata_dict.values())[456])

Total samples in dataset: 204978

Example of a sample:
{'answer_key': 'K01_V002-146900',
 'fps': '30.0',
 'frame_id': '048',
 'frame_order': '4407',
 'img_link': 'https://storage.cloud.google.com/aic_2025/kfs/Keyframes_K01/K01_V002/048.jpg',
 'publish_date': '03/10/2024',
 'time_order': '146900',
 'vector': [],
 'vid_link': 'https://storage.cloud.google.com/aic_2025/vids/Videos_K01/K01_V002.mp4',
 'video_id': 'K01_V002',
 'youtube_link': 'https://youtube.com/watch?v=-j_pLhi87I8&t=146900ms'}


**DATASET RETUNING**

In [27]:
dataset = torch.load('dataset-aic2025-no-feature-b1.pt') # Do the same to the no feature one (save without 'wfeat' in name)
http_prefix = 'https://storage.cloud.google.com/aic_2025/'
for entity in list(dataset.values()):
    entity['img_link'] = http_prefix + entity['img_key']
    entity['vid_link'] = http_prefix + entity['vid_key']
    entity.pop('img_key')
    entity.pop('vid_key')

torch.save(dataset, 'aic_2025-b1.pt')

In [28]:
print(f'Total samples in dataset: {len(dataset)}')
print('\nExample of a sample:')
pprint(list(dataset.values())[456])

Total samples in dataset: 177321

Example of a sample:
{'answer_key': 'L21_V002-580200',
 'fps': '30.0',
 'frame_id': '150',
 'frame_order': '17406',
 'img_link': 'https://storage.cloud.google.com/aic_2025/kfs/Keyframes_L21/L21_V002/150.jpg',
 'publish_date': '02/08/2024',
 'time_order': '580200',
 'vector': [],
 'vid_link': 'https://storage.cloud.google.com/aic_2025/vids/Videos_L21/L21_V002.mp4',
 'video_id': 'L21_V002',
 'youtube_link': 'https://youtube.com/watch?v=qqC4FLMNLdI&t=580200ms'}
